In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [16]:
lines = sc.textFile('Desktop/input.txt')\
.flatMap(lambda l: l.split(" "))\
.filter(lambda w: w != "")\
.map(lambda w: w.replace("(","").replace(")",""))\
.map(lambda w: (w,1))\
.reduceByKey(lambda a,b: a+b)\
.sortBy(lambda x: x[1], False)

lines.take(10)

[('a', 12),
 ('és', 5),
 ('A', 5),
 ('amerikai', 3),
 ('Wilder', 3),
 ('Van,', 2),
 ('aki', 2),
 ('filmje', 2),
 ('az', 2),
 ('Amerikai', 2)]

In [28]:
from pyspark.sql import Row

columns = ["language", "users_count"]
data  = [("Java", "20000"), ("Python", "100000"),("Scala", "3000")]
rdd = sc.parallelize(data)

dfFromRDD1 = rdd.toDF()
dfFromRDD1.printSchema()

dfFromRDD2 = spark.createDataFrame(rdd).toDF(*columns)
dfFromRDD2.printSchema()

rowData = map(lambda x: Row(*x), data)
dfFromRDD3 = spark.createDataFrame(rowData,columns)
dfFromRDD3.printSchema()

#Sema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
data2 = [
    ("James", "Smith", "M", 3000),
    ("Michael","Rose","M",4000),
    ("Angel","Rose","F",5000),
    ("Michael","Rose","M",4000),
]

schema = StructType([\
    StructField("firstname",StringType(), True),\
    StructField("lastname",StringType(), True),\
    StructField("gender",StringType(), True),\
    StructField("salary",IntegerType(), True)\
])

df = spark.createDataFrame(data=data2, schema=schema)
df.printSchema()
df.show()

schema = StructType([\
    StructField("name",StructType([
        StructField("firstname",StringType(), True),\
        StructField("lastname",StringType(), True),\
    ])),
    StructField("gender",StringType(), True),\
])

data = [
    (("James","Smith"),"M"),
    (("Anna","Roe"),"F"),
]

df2 = spark.createDataFrame(data=data, schema=schema)
df2.printSchema()
df2.show()




root
 |-- _1: string (nullable = true)
 |-- _2: string (nullable = true)

root
 |-- language: string (nullable = true)
 |-- users_count: string (nullable = true)

root
 |-- language: string (nullable = true)
 |-- users_count: string (nullable = true)

root
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)

+---------+--------+------+------+
|firstname|lastname|gender|salary|
+---------+--------+------+------+
|    James|   Smith|     M|  3000|
|  Michael|    Rose|     M|  4000|
|    Angel|    Rose|     F|  5000|
|  Michael|    Rose|     M|  4000|
+---------+--------+------+------+

root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- gender: string (nullable = true)

+--------------+------+
|          name|gender|
+--------------+------+
|{James, Smith}|     M|
|   {Anna, Roe}|     F|
+-----------

In [29]:
df3 = spark.read.csv("Downloads/zipcodes.csv")

In [36]:
df.select("firstname","lastname").cache().show()
df.select(df.firstname, df.lastname).cache().show()

df2.select("name").show()
df2.select("name.firstname","name.lastname").show()
df2.select("name.*").show()

+---------+--------+
|firstname|lastname|
+---------+--------+
|    James|   Smith|
|  Michael|    Rose|
|    Angel|    Rose|
|  Michael|    Rose|
+---------+--------+

+---------+--------+
|firstname|lastname|
+---------+--------+
|    James|   Smith|
|  Michael|    Rose|
|    Angel|    Rose|
|  Michael|    Rose|
+---------+--------+

+--------------+
|          name|
+--------------+
|{James, Smith}|
|   {Anna, Roe}|
+--------------+

+---------+--------+
|firstname|lastname|
+---------+--------+
|    James|   Smith|
|     Anna|     Roe|
+---------+--------+

+---------+--------+
|firstname|lastname|
+---------+--------+
|    James|   Smith|
|     Anna|     Roe|
+---------+--------+



In [40]:
df.filter(df.gender == 'M').show()

df.filter("gender == 'F'").show()

df.filter((df.gender == 'M') & (df.salary == 3000)).show()

df2.filter(df2.name.firstname == "Anna").show()

+---------+--------+------+------+
|firstname|lastname|gender|salary|
+---------+--------+------+------+
|    James|   Smith|     M|  3000|
|  Michael|    Rose|     M|  4000|
|  Michael|    Rose|     M|  4000|
+---------+--------+------+------+

+---------+--------+------+------+
|firstname|lastname|gender|salary|
+---------+--------+------+------+
|    Angel|    Rose|     F|  5000|
+---------+--------+------+------+

+---------+--------+------+------+
|firstname|lastname|gender|salary|
+---------+--------+------+------+
|    James|   Smith|     M|  3000|
+---------+--------+------+------+

+-----------+------+
|       name|gender|
+-----------+------+
|{Anna, Roe}|     F|
+-----------+------+



In [42]:
df.distinct().show()

df.dropDuplicates(["lastname"]).show()

+---------+--------+------+------+
|firstname|lastname|gender|salary|
+---------+--------+------+------+
|    Angel|    Rose|     F|  5000|
|    James|   Smith|     M|  3000|
|  Michael|    Rose|     M|  4000|
+---------+--------+------+------+

+---------+--------+------+------+
|firstname|lastname|gender|salary|
+---------+--------+------+------+
|  Michael|    Rose|     M|  4000|
|    James|   Smith|     M|  3000|
+---------+--------+------+------+



In [43]:
df.sort("firstname","lastname").show()
df.orderBy("firstname","lastname").show()
df.sort(df.firstname.desc(),df.lastname.asc()).show()

+---------+--------+------+------+
|firstname|lastname|gender|salary|
+---------+--------+------+------+
|    Angel|    Rose|     F|  5000|
|    James|   Smith|     M|  3000|
|  Michael|    Rose|     M|  4000|
|  Michael|    Rose|     M|  4000|
+---------+--------+------+------+

+---------+--------+------+------+
|firstname|lastname|gender|salary|
+---------+--------+------+------+
|    Angel|    Rose|     F|  5000|
|    James|   Smith|     M|  3000|
|  Michael|    Rose|     M|  4000|
|  Michael|    Rose|     M|  4000|
+---------+--------+------+------+

+---------+--------+------+------+
|firstname|lastname|gender|salary|
+---------+--------+------+------+
|  Michael|    Rose|     M|  4000|
|  Michael|    Rose|     M|  4000|
|    James|   Smith|     M|  3000|
|    Angel|    Rose|     F|  5000|
+---------+--------+------+------+



In [44]:
df.createOrReplaceTempView("EMP")
spark.sql("select * from EMP order by firstname asc").show()

+---------+--------+------+------+
|firstname|lastname|gender|salary|
+---------+--------+------+------+
|    Angel|    Rose|     F|  5000|
|    James|   Smith|     M|  3000|
|  Michael|    Rose|     M|  4000|
|  Michael|    Rose|     M|  4000|
+---------+--------+------+------+



In [ ]:
from pyspark.sql.functions import sum,avg,max

df.groupBy("gender").sum("salary").show()
df.groupBy("gender").count().show()

df.groupBy("gender").agg(
    sum("salary").alias("sum_sal"), \
    avg("salary").alias("avg_sal"), \
    max("salary").alias("max_sal")
).show()


In [49]:
import pyspark.sql.functions as F

empDF = df.withColumn("emp_dept_id", (F.col("salary")/100))

dept = [
    ("Finance",10),\
    ("Marketing", 20), \
    ("Sales",30), \
    ("IT", 40)
]
deptColumns = ["dept_name", "dept_id"]

deptDF = spark.createDataFrame(data=dept, schema=deptColumns)

empDF.join(deptDF, empDF.emp_dept_id == deptDF.dept_id, "inner"  ).show()

empDF.join(deptDF, empDF.emp_dept_id == deptDF.dept_id, "full"  ).show()

empDF.join(deptDF, empDF.emp_dept_id == deptDF.dept_id, "left"  ).show()


+---------+--------+------+------+-----------+---------+-------+
|firstname|lastname|gender|salary|emp_dept_id|dept_name|dept_id|
+---------+--------+------+------+-----------+---------+-------+
|    James|   Smith|     M|  3000|       30.0|    Sales|     30|
|  Michael|    Rose|     M|  4000|       40.0|       IT|     40|
|  Michael|    Rose|     M|  4000|       40.0|       IT|     40|
+---------+--------+------+------+-----------+---------+-------+

+---------+--------+------+------+-----------+---------+-------+
|firstname|lastname|gender|salary|emp_dept_id|dept_name|dept_id|
+---------+--------+------+------+-----------+---------+-------+
|    Angel|    Rose|     F|  5000|       50.0|     null|   null|
|     null|    null|  null|  null|       null|  Finance|     10|
|    James|   Smith|     M|  3000|       30.0|    Sales|     30|
|  Michael|    Rose|     M|  4000|       40.0|       IT|     40|
|  Michael|    Rose|     M|  4000|       40.0|       IT|     40|
|     null|    null|  nu

In [50]:
from pyspark.sql.functions import collect_list,collect_set,sum,mean,max,first,last,min

df.select(collect_list("salary")).show()
df.select(collect_set("salary")).show()
df.select(first("salary")).show()
df.select(last("salary")).show()
df.select(max("salary")).show()
df.select(min("salary")).show()
df.select(mean("salary")).show()
df.select(sum("salary")).show()


+--------------------+
|collect_list(salary)|
+--------------------+
|[3000, 4000, 5000...|
+--------------------+

+-------------------+
|collect_set(salary)|
+-------------------+
| [3000, 5000, 4000]|
+-------------------+

+-------------+
|first(salary)|
+-------------+
|         3000|
+-------------+

+------------+
|last(salary)|
+------------+
|        4000|
+------------+

+-----------+
|max(salary)|
+-----------+
|       5000|
+-----------+

+-----------+
|min(salary)|
+-----------+
|       3000|
+-----------+

+-----------+
|avg(salary)|
+-----------+
|     4000.0|
+-----------+

+-----------+
|sum(salary)|
+-----------+
|      16000|
+-----------+

